# 01. Agentic AI 기초: 호출 한 번에서 Agent까지

이 과정이 끝나면 만들게 될 것은 사내 문서를 읽고 근거를 대며 답하는 RAG Agent입니다. 그 Agent도 이 노트북에서 만드는 것과 같은 부품(모델, 도구, 지침, 상태, 종료 조건)으로 되어 있습니다. 오늘은 사내 식당 메뉴 같은 작은 주제로 그 부품을 하나씩 얹어 봅니다.

## 학습 목표

- 단순 모델 호출, 고정 workflow, Agent의 차이를 "누가 다음 행동을 정하는가"로 설명한다.
- 도구를 쓸 때 모델은 **요청**만 하고 **실행**은 애플리케이션이 한다는 것을 출력으로 확인한다.
- `create_deep_agent` 결과의 메시지 목록을 한 줄씩 읽고 무슨 일이 있었는지 말한다.
- 없는 도구와 끝나지 않는 도구를 일부러 만들어 Agent가 어떻게 실패하는지 읽는다.

## 수업 시간표

| 절 | 내용 | 성격 | 분 |
|---|---|---|---:|
| 1 | 왜 Agent인가: 호출 한 번, 고정 순서, 다음 행동을 고르는 Agent | 설명 | 6 |
| 2 | 같은 질문 세 번: 도구 없이, 도구 하나, Agent | 실습 | 9 |
| 3 | 세 결과의 차이와 Agent의 다섯 구성요소 | 설명 | 5 |
| 4 | 시스템 프롬프트 바꾸기 | 실습 | 5 |
| 5 | 메시지가 상태다: 메시지를 한 줄씩 출력하기 | 실습 | 11 |
| 6 | 종료 조건과 실패, 없는 도구 | 설명 + 실습 | 5 |
| 7 | 끝나지 않는 도구 | 실습 | 6 |
| - | 직접 해보기: 내 업무로 같은 세 단계 | 내 것 | 10 |
| - | 정리와 확인 질문 | 설명 | 3 |

## 이 노트북의 약속

Day01에서는 `StateGraph`로 노드와 엣지를 직접 이어 에이전트 루프를 만들어 보았습니다. 이 노트북은 그 루프를 **다시 만들지 않습니다.** Deep Agents의 `create_deep_agent`가 같은 종류의 순환(모델 호출, 도구 실행, 모델 호출)을 대신 조립해 주기 때문입니다. 우리는 조립된 결과를 실행하고 읽습니다.

코드 셀 맨 윗줄의 표지는 셀이 하는 일을 알려 줍니다.

- `# [따라하기]` 그대로 실행하고 출력을 관찰합니다.
- `# [바꿔보기]` 값 하나를 바꿉니다. **실행하기 전에** 무엇이 달라질지 먼저 적어 보세요.
- `# [깨뜨리기]` 일부러 실패시키고 오류를 읽습니다.
- `# [내 것 만들기]` 예시가 채워져 있어 그대로 돕니다. 내 업무 내용으로 바꿔 다시 실행합니다.

모델 호출이 필요한 자리에는 모두 **실제 Claude**가 들어옵니다. 모델이 도구를 부를지는 우리가 정하지 않으므로, 셀 아래의 "관찰" 글은 정해진 정답이 아니라 **이번 출력을 읽는 방법**입니다. 유료 셀은 실행 직전에 `announce`가 호출 규모를 한 줄로 알려 줍니다.

In [ ]:
from rag_lab.settings import key_status, require_key

require_key()  # 키가 없으면 여기서 한국어 안내와 함께 멈춥니다.
print(key_status())

키 값은 출력되지 않고 `설정됨`/`미설정`과 모델 이름, effort 같은 설정만 나옵니다. `미설정`이면 위 셀이 멈추므로 `Day02_new/.env.example`을 보고 `.env`를 직접 채운 뒤 다시 시작하세요.

## 1절. 호출 한 번, 고정 순서, 다음 행동을 고르는 Agent (6분)

같은 질문을 처리하는 방법은 세 가지로 나뉩니다. 갈라지는 기준은 하나, **누가 다음 행동을 정하는가**입니다.

| 방식 | 다음 행동을 정하는 쪽 | 예 |
|---|---|---|
| 모델 호출 한 번 | 정할 행동이 없다. 입력 하나에 출력 하나 | 질문을 보내고 문장을 받는다 |
| 고정 workflow | 사람이 코드로 순서를 쓴다 | 요일로 메뉴 사전을 찾고, 그 결과를 문장에 끼운다 |
| Agent | 모델이 매번 고른다 | 메뉴를 찾을지, 바로 답할지, 도구를 또 부를지 모델이 정한다 |

고정 workflow는 예측하기 쉽고 틀려도 어느 줄인지 바로 보입니다. 대신 분기가 늘면 코드가 늡니다. Agent는 분기를 모델에게 맡겨 코드를 줄이는 대신 호출 수와 결과가 실행마다 달라집니다. 어느 쪽이 낫다는 답은 없고 일의 성격이 고릅니다.

먼저 가장 단순한 형태입니다. 아래 셀은 질문을 모델에 한 번 보냅니다. 사내 식당 메뉴는 모델이 알 길이 없는 정보입니다. `make_model()`은 `Day02_new/.env`의 키와 모델 설정으로 Claude 호출 객체를 만들어 줍니다. 요청 설정을 한곳에 모으고 사용량을 기록하는 일만 하며 모델의 판단은 바꾸지 않습니다.

In [ ]:
# [따라하기] 유료, 모델 호출 약 1회
from rag_lab.settings import announce, make_model

question = "이번 주 월요일 사내 식당 메뉴가 뭐예요?"
announce("도구 없이 묻기", model_calls=1)
reply = make_model().invoke(question)
print(reply.text)
print("tool_calls:", reply.tool_calls)

**관찰.** `tool_calls`가 비어 있는지 보세요. 모델은 아무것도 실행하지 않았고 문자열만 돌려줬습니다. 본문에서는 모델이 메뉴를 **모른다고 하는지**, 그럴듯하게 **지어내는지**를 읽습니다. 답 글은 `reply.content`가 아니라 `reply.text`로 꺼냅니다. `content`는 문자열이 아니라 블록 목록일 수 있기 때문입니다.

이번에는 값 하나만 바꿉니다. 실행하기 전에 아래 셀의 예측 줄에 생각을 적어 보세요.

In [ ]:
# [바꿔보기] 유료, 모델 호출 약 1회
# 바꿀 값: question
# 예측을 먼저 적으세요: 이번에도 모른다고 할까요? 도구가 필요한 질문과 필요 없는 질문은 무엇이 다를까요?
question = "대한민국의 수도는 어디인가요?"
announce("도구 없이 묻기, 다른 질문", model_calls=1)
reply = make_model().invoke(question)
print(reply.text)
print("tool_calls:", reply.tool_calls)

**관찰.** 이번에는 모델이 바로 답했는지 확인하세요. 모델 안에 이미 있는 지식으로 풀리는 질문은 도구가 필요 없고, 회사 안의 최신 정보는 모델 밖에 있습니다. 이 차이가 "도구를 줄 이유"입니다.

## 2절. 같은 질문, 도구 하나

도구를 주면 모델은 문장 대신 **도구 요청**을 돌려줄 수 있습니다. 중요한 점은 모델이 도구를 직접 실행하지 않는다는 것입니다. 모델은 "`get_menu`를 `월요일`로 불러 주세요"라는 요청만 만들고, 실행은 우리 프로그램이 합니다. `@tool`은 파이썬 함수에 모델에게 보여 줄 이름, 설명, 인자 정의를 붙여 줍니다. 모델이 도구를 고르는 단서는 이 설명과 인자 정의이고 함수 본문은 모델에게 보이지 않습니다.

In [ ]:
# [따라하기] 유료, 모델 호출 약 1회
from langchain_core.tools import tool


@tool
def get_menu(day: str) -> str:
    """요일을 받아 그날의 사내 식당 메뉴를 돌려준다."""
    menus = {"월요일": "김치찌개", "화요일": "비빔밥", "수요일": "돈가스"}
    return menus.get(day, "그 요일의 메뉴 정보가 없습니다.")


question = "이번 주 월요일 사내 식당 메뉴가 뭐예요?"
announce("도구 하나 묶기", model_calls=1)
reply = make_model().bind_tools([get_menu]).invoke(question)
print("text:", repr(reply.text))
print("tool_calls:", reply.tool_calls)

**관찰.** `tool_calls`에 도구 이름과 인자(`day`)가 들어 있나요? 그리고 메뉴 글자(김치찌개 같은)는 **아직 어디에도 없습니다.** 모델은 요청만 했고 `get_menu`는 한 번도 실행되지 않았습니다. 이 요청을 실행하는 일은 우리 몫입니다.

In [ ]:
# [따라하기] 무료, 모델 호출 0회
if reply.tool_calls:
    tool_message = get_menu.invoke(reply.tool_calls[0])
    print(type(tool_message).__name__, "| 내용:", tool_message.content)
else:
    print("이번에는 모델이 도구를 요청하지 않았습니다. 바로 위 셀을 다시 실행해 보세요.")

**관찰.** 함수는 우리 프로그램이 실행했고 결과는 `ToolMessage`로 돌아옵니다. `tool_calls`가 비어 있어 안내 문구가 나왔다면 그것도 사실입니다. 모델이 도구를 쓸지는 모델이 정하므로 매번 요청하는 것은 아닙니다.

이번에는 메뉴와 상관없는 질문을 줍니다. 예측부터 적으세요.

In [ ]:
# [바꿔보기] 유료, 모델 호출 약 1회
# 바꿀 값: question
# 예측을 먼저 적으세요: 모델이 메뉴 도구를 부를까요? 부른다면 어떤 인자로 부를까요?
question = "통근 버스는 몇 시에 출발하나요?"
announce("도구 하나 묶기, 관련 없는 질문", model_calls=1)
reply = make_model().bind_tools([get_menu]).invoke(question)
print("text:", repr(reply.text))
print("tool_calls:", reply.tool_calls)

**관찰.** `tool_calls`가 비었다면 모델은 메뉴 도구가 이 질문에 맞지 않는다고 판단한 것입니다. 모델이 도구를 **고른다**는 말은 이런 뜻입니다. 도구를 받았다고 항상 쓰는 것이 아니고, 쓸지 말지의 판단이 곧 "다음 행동을 정한다"입니다.

## 3절. Agent: 왕복을 harness가 맡는다

지금까지는 요청과 실행 사이를 우리가 손으로 이었습니다. 요청을 받아 실행하고 결과를 다시 모델에게 주고 최종 답을 받는 일을 매번 쓰면 번거롭습니다. `create_deep_agent`는 이 왕복을 대신 돌려 주는 harness입니다. 우리는 모델과 도구만 주고 질문을 넣습니다.

In [ ]:
# [따라하기] 유료, 모델 호출 약 2회
from deepagents import create_deep_agent

question = "이번 주 월요일 사내 식당 메뉴가 뭐예요?"
announce("Agent 첫 실행", model_calls=2)
agent = create_deep_agent(model=make_model(), tools=[get_menu])
result = agent.invoke({"messages": [{"role": "user", "content": question}]})
print(result["messages"][-1].text)
print("메시지 수:", len(result["messages"]))

**관찰.** 최종 답이 `result["messages"][-1].text`에 있습니다. 앞 절의 두 단계(요청, 실행)를 우리가 한 줄도 쓰지 않았는데 답에 메뉴가 들어 있나요? 메시지 수는 이번 실행에서 오간 기록의 개수이며 4 이상일 수도 있습니다. 모델이 도구를 더 불렀다면 그만큼 늘어납니다.

Agent를 이루는 부품은 다섯 가지로 정리할 수 있습니다. **모델**(판단), **도구**(바깥 세계에 닿는 손), **지침**(시스템 프롬프트), **상태**(지금까지의 메시지), **종료 조건**(언제 멈추는가)입니다. 아래 셀은 `create_deep_agent`의 인자 목록에서 각 부품에 해당하는 이름을 찾습니다.

In [ ]:
# [따라하기] 무료, 모델 호출 0회
import inspect

params = inspect.signature(create_deep_agent).parameters
roles = {
    "모델": ["model"],
    "도구": ["tools", "subagents"],
    "지침": ["system_prompt", "skills", "memory"],
    "상태": ["state_schema", "backend", "checkpointer", "store"],
}
for role, names in roles.items():
    print(f"{role} -> " + ", ".join(f"{n}{'' if n in params else '(없음)'}" for n in names))
print("종료 조건: recursion_limit이 인자 목록에 있는가 ->", "recursion_limit" in params)

**관찰.** 모델, 도구, 지침, 상태에 해당하는 인자는 만들 때 넘깁니다. `(없음)`이 붙은 이름이 있다면 설치된 버전에서 이름이 다른 것이니 출력을 따릅니다. 종료 조건만 인자 목록에 없습니다. 종료 조건은 만들 때가 아니라 **실행할 때** `config`의 `recursion_limit`으로 정합니다. 7절에서 직접 깨뜨려 봅니다.

## 4절. 시스템 프롬프트 바꾸기

지침은 모델이 매번 읽는 "역할 설명"입니다. 같은 도구, 같은 질문에 지침만 바꿔 답의 모양이 달라지는지 봅니다.

In [ ]:
# [따라하기] 유료, 모델 호출 약 2회
announce("시스템 프롬프트 한 문장", model_calls=2)
agent = create_deep_agent(
    model=make_model(), tools=[get_menu],
    system_prompt="당신은 사내 생활 안내원입니다. 한 문장으로 답합니다.",
)
result = agent.invoke({"messages": [{"role": "user", "content": question}]})
print(result["messages"][-1].text)

**관찰.** 앞 절 3의 답과 길이와 말투를 비교하세요. `system_prompt`는 모델에게 가는 지침에 더해지고, harness가 붙이는 기본 지침을 완전히 대체하지는 않을 수 있습니다. 모델이 실제로 받은 지침 전체는 03에서 출력해 확인합니다.

In [ ]:
# [바꿔보기] 유료, 모델 호출 약 2회
# 바꿀 값: system_prompt
# 예측을 먼저 적으세요: 답의 형식과 도구 호출 중 무엇이 바뀔까요? 메시지 수도 달라질까요?
announce("시스템 프롬프트를 표로", model_calls=2)
agent = create_deep_agent(
    model=make_model(), tools=[get_menu],
    system_prompt="당신은 사내 생활 안내원입니다. 메뉴를 표로 답합니다.",
)
result = agent.invoke({"messages": [{"role": "user", "content": question}]})
print(result["messages"][-1].text)
print("메시지 수:", len(result["messages"]))

**관찰.** 답의 **형식**이 바뀌었나요? 바뀐 것은 지침이므로 도구를 부른 횟수(메시지 수)는 대체로 그대로일 것이라고 예측했다면 실제와 맞는지 확인하세요. 지침은 모델이 도구를 부르는 판단에는 약하게, 말하는 방식에는 강하게 작용하는 경우가 많지만 이것도 실행마다 다를 수 있습니다.

## 5절. 메시지가 상태다

Agent의 "상태"는 사실상 **메시지 목록**입니다. 사람 메시지, 모델의 도구 요청, 도구 결과, 모델의 최종 답이 순서대로 쌓입니다. 모델은 매번 이 목록 전체를 읽고 다음 행동을 고릅니다. 그러니 이 목록을 읽을 수 있으면 Agent가 무엇을 했는지 설명할 수 있습니다. 먼저 `for` 문 하나로 한 줄씩 찍어 봅니다.

In [ ]:
# [따라하기] 무료, 모델 호출 0회
for m in result["messages"]:
    print(type(m).__name__, getattr(m, "tool_calls", None), m.text[:60])

**관찰.** 보통 Human, AI(도구 요청), Tool, AI(답)의 순서입니다. 도구를 요청하는 AI 줄은 `text`가 비고 `tool_calls`에 이름과 인자가 있습니다. Tool 줄의 `text`는 도구가 돌려준 결과이고 마지막 AI 줄이 최종 답입니다. 이번에는 도구를 하나 더 주고 한 질문에 두 가지를 묻습니다.

In [ ]:
# [바꿔보기] 유료, 모델 호출 약 3회
# 바꿀 값: tools (get_bus_time 추가)
# 예측을 먼저 적으세요: 메시지 줄이 몇 개 늘까요? 한 AI 줄에 도구 요청이 둘 들어 있을까요?
@tool
def get_bus_time(route: str) -> str:
    """통근 버스 노선 이름을 받아 출발 시각을 돌려준다."""
    return {"판교": "오후 6시", "수원": "오후 6시 30분"}.get(route, "그 노선 정보가 없습니다.")


announce("도구 둘, 질문 둘", model_calls=3)
agent = create_deep_agent(model=make_model(), tools=[get_menu, get_bus_time])
result = agent.invoke({"messages": [{"role": "user", "content": "월요일 식당 메뉴와 판교행 통근 버스 출발 시각을 알려 주세요."}]})
for m in result["messages"]:
    print(type(m).__name__, getattr(m, "tool_calls", None), m.text[:60])

**관찰.** 줄 수를 앞 셀과 비교하세요. AI 줄 하나에 `tool_calls`가 둘 들어 있으면 모델이 두 도구를 **한꺼번에** 요청한 것이고, AI 줄이 둘로 나뉘면 하나씩 차례로 요청한 것입니다. 어느 쪽이든 정답은 아니며 이번 실행에서 모델이 고른 방식입니다.

줄마다 `print`를 쓰다 보니 읽기 불편합니다. 같은 반복을 이제 함수로 묶겠습니다. 이 노트북에서 **처음 정의하는** 함수이고 앞의 `for` 문과 같은 내용에 도구 실행 상태(`status`) 칸을 더한 것뿐입니다.

In [ ]:
# [따라하기] 무료, 모델 호출 0회
def show(messages):
    for i, m in enumerate(messages):
        calls = ", ".join(f"{c['name']}({c['args']})" for c in getattr(m, "tool_calls", None) or []) or "-"
        status = getattr(m, "status", "-")
        print(f"{i:2d} {type(m).__name__:13s} status={status:7s} 도구요청={calls} | {m.text[:50]!r}")


show(result["messages"])

**관찰.** 바로 앞 셀의 `for` 출력과 내용이 같아야 합니다. `status` 칸은 Tool 줄에서만 의미가 있습니다. 도구가 정상이면 `success`, 실패했으면 `error`입니다. 다음 절에서 `error`를 일부러 만들어 봅니다.

## 6절. 종료 조건과 실패

Agent는 모델이 "이제 답한다"고 고를 때 멈춥니다. 그래서 두 가지 문제가 생깁니다. **없는 도구**를 요청하거나, 도구 결과가 마음에 들지 않아 **끝없이 다시 부르는** 경우입니다. 그리고 가장 조용한 세 번째 실패가 있습니다. 도구를 쓰지 않고 그럴듯하게 답하는 것입니다. 이때는 예외도 없이 정상 종료합니다.

없는 도구 요청이 들어오면 harness는 프로그램을 죽이지 않고 **오류를 `ToolMessage`로 만들어 모델에게 돌려줍니다.** 실제 모델은 없는 도구를 좀처럼 요청하지 않으므로, 아래 첫 셀은 **손으로 만든** 도구 요청을 직접 넣어 이 구조를 확인합니다. 손으로 메시지를 만드는 것은 이 `[깨뜨리기]` 셀 안에서뿐입니다. 모델은 부르지 않습니다.

In [ ]:
# [깨뜨리기] 무료, 모델 호출 0회
# 손으로 만든 요청: 실제 모델이 낸 것이 아니라 우리가 AIMessage를 직접 만들어 상태에 넣는다.
from uuid import uuid4

from langchain_core.messages import AIMessage, HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

hand_made = AIMessage(content="", id="hand-made",
                      tool_calls=[{"name": "get_price", "args": {"menu": "김치찌개"}, "id": "call-hand-1"}])
checked = create_deep_agent(model=make_model(), tools=[get_menu], checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": uuid4().hex}}
checked.update_state(config, {"messages": [HumanMessage("메뉴 가격이 얼마예요?"), hand_made]}, as_node="model")
messages = checked.invoke(None, config, interrupt_before=["model"])["messages"]
show(messages)
print(messages[-1].content)

**관찰.** 마지막 `ToolMessage`의 `status`가 `error`이고 내용에 "유효한 도구가 아니다"라는 취지의 문구와 **쓸 수 있는 도구 이름 목록**이 있는지 읽으세요. 프로그램은 죽지 않았고, 오류가 다음 모델 호출의 입력으로 돌아가는 구조입니다. 이제 실제 모델에게 없는 도구를 쓰라고 시킵니다.

In [ ]:
# [깨뜨리기] 유료, 모델 호출 약 2회
announce("없는 도구를 쓰라고 시키기", model_calls=2)
agent = create_deep_agent(model=make_model(), tools=[get_menu])
result = agent.invoke({"messages": [{"role": "user", "content": "get_price 도구로 김치찌개 가격을 찾아 주세요."}]})
show(result["messages"])
asked = [c["name"] for m in result["messages"] for c in (getattr(m, "tool_calls", None) or [])]
print("모델이 요청한 도구 이름:", asked)
print("get_price를 요청했는가:", "get_price" in asked)

**관찰.** `get_price`를 요청한 줄이 **있는지** 먼저 봅니다. 요청하지 않고 "그런 도구는 없다"고 답했다면, 이 실패는 예외 없이 정상 종료한 것이고 그것도 실패의 한 모양입니다. 모델이 다른 도구나 파일 도구로 돌려 푸는 경우도 있습니다. 어느 쪽이었는지는 이번 출력이 말해 줍니다.

## 7절. 끝나지 않는 도구

이번에는 항상 "아직 안 끝났다"고만 말하는 도구를 줍니다. 모델이 포기하지 않으면 도구 실행과 모델 호출이 끝없이 이어집니다. 종료 조건은 `recursion_limit`이 맡고 한도를 넘으면 `GraphRecursionError`가 납니다. 도구 안의 전역 목록으로 **도구가 몇 번 실행됐는지** 셉니다.

In [ ]:
# [깨뜨리기] 유료, 모델 호출 약 3회
from langgraph.errors import GraphRecursionError

runs = []


@tool
def poll_status(job: str) -> str:
    """작업 상태를 확인한다. 완료될 때까지 같은 인자로 다시 호출해야 한다."""
    runs.append(job)
    return "상태: 대기 중. 아직 완료되지 않았습니다. 완료될 때까지 다시 호출하세요."


announce("끝나지 않는 도구, recursion_limit 6", model_calls=3)
agent = create_deep_agent(model=make_model(), tools=[poll_status])
try:
    out = agent.invoke({"messages": [{"role": "user", "content": "작업 job-1이 완료될 때까지 poll_status로 확인하고 알려 주세요."}]},
                       config={"recursion_limit": 6})
    print("한도 안에서 끝났습니다. 최종 답:", out["messages"][-1].text[:80])
except GraphRecursionError as error:
    print(type(error).__name__, "|", str(error)[:80])
print("도구 실행 횟수:", len(runs))

**관찰.** `GraphRecursionError`가 나왔다면 모델이 대기 중이라는 결과를 받고도 계속 부르다가 한도가 끊은 것입니다. 한도 안에서 끝났다면 모델이 스스로 포기하고 답한 것이며 이것도 모델의 판단이라 다음 실행에서 달라질 수 있습니다. 어느 쪽이든 **끊어 줄 장치가 있는가**가 요점입니다. 기본 한도는 매우 커서 아무것도 안 하면 비용이 오래 새어 나갑니다. 한도를 줄여 다시 해 봅니다.

In [ ]:
# [바꿔보기] 유료, 모델 호출 약 2회
# 바꿀 값: recursion_limit (6을 4로)
# 예측을 먼저 적으세요: 도구 실행은 몇 번일까요? 한도 숫자와 도구 실행 횟수가 같을까요?
runs.clear()
announce("끝나지 않는 도구, recursion_limit 4", model_calls=2)
agent = create_deep_agent(model=make_model(), tools=[poll_status])
try:
    out = agent.invoke({"messages": [{"role": "user", "content": "작업 job-1이 완료될 때까지 poll_status로 확인하고 알려 주세요."}]},
                       config={"recursion_limit": 4})
    print("한도 안에서 끝났습니다. 최종 답:", out["messages"][-1].text[:80])
except GraphRecursionError as error:
    print(type(error).__name__, "|", str(error)[:80])
print("도구 실행 횟수:", len(runs))

**관찰.** 앞 셀의 실행 횟수와 비교하세요. 한도가 작을수록 도구 실행이 적습니다. 다만 한 번의 "모델 호출 + 도구 실행"이 한도를 여러 단계 쓰기 때문에 **한도와 도구 실행 횟수가 1:1은 아닙니다.** 이 관계가 05에서 호출 예산을 따로 세는 이유입니다.

## 직접 해보기

내 업무로 같은 세 단계(도구 없이, 도구 하나, Agent)를 돌려 봅니다. 두 셀 모두 예시가 채워져 있어 그대로 실행되고, 값을 비우면 안내만 나오고 모델은 부르지 않습니다. 예시를 내 업무의 값으로 바꿔 다시 실행하세요. 도구 설명과 인자 이름도 내 업무에 맞게 함께 바꿉니다. 모델은 그 설명을 보고 도구를 고릅니다.

In [ ]:
# [내 것 만들기] 유료, 모델 호출 약 4회
MY_DATA = {"김민수": "12일", "이서연": "7일", "박지훈": "3일"}  # 예시입니다. 내 업무의 조회용 값으로 바꾸세요. (이름: 잔여 연차)
MY_QUESTION = "김민수 님의 남은 연차는 며칠인가요?"  # 예시입니다. 내 업무 질문으로 바꾸세요.


@tool
def my_lookup(name: str) -> str:
    """이름을 받아 그 사람의 남은 연차를 돌려준다."""  # 바꿀 곳: 이 설명과 인자 이름 name
    return MY_DATA.get(name, "그 이름의 정보가 없습니다.")


if not MY_QUESTION.strip() or not MY_DATA:
    print("MY_DATA와 MY_QUESTION이 비어 있습니다. 내 업무의 조회용 값 몇 개와 질문 한 문장을 적고 다시 실행하세요.")
else:
    announce("내 업무 세 단계", model_calls=4)
    plain = make_model().invoke(MY_QUESTION)
    print("1) 도구 없이:", plain.text[:100])
    asked = make_model().bind_tools([my_lookup]).invoke(MY_QUESTION)
    print("2) 도구 하나, 요청만:", asked.tool_calls)
    mine = create_deep_agent(model=make_model(), tools=[my_lookup])
    mine_result = mine.invoke({"messages": [{"role": "user", "content": MY_QUESTION}]})
    print("3) Agent:", mine_result["messages"][-1].text[:100])

**관찰.** 세 결과를 나란히 읽으세요. 1)은 모델이 모르는 것을 어떻게 다뤘는지, 2)는 요청에 내 도구 이름과 인자가 제대로 들어 있는지, 3)은 답에 도구가 돌려준 값이 들어 있는지 봅니다. 내 업무의 값이 사전 하나로 표현되지 않는다면 그 어려움이 04부터 다루는 검색의 출발점입니다.

In [ ]:
# [내 것 만들기] 유료, 모델 호출 약 2회
MY_SYSTEM_PROMPT = "당신은 인사팀 안내원입니다. 숫자는 그대로 쓰고 한 문장으로 답합니다."  # 예시입니다. 내 업무 말투로 바꾸세요.

if not MY_SYSTEM_PROMPT.strip():
    print("MY_SYSTEM_PROMPT가 비어 있습니다. 내 업무에 맞는 말투와 규칙을 한두 문장으로 적고 다시 실행하세요.")
else:
    announce("내 시스템 프롬프트", model_calls=2)
    mine = create_deep_agent(model=make_model(), tools=[my_lookup], system_prompt=MY_SYSTEM_PROMPT)
    mine_result = mine.invoke({"messages": [{"role": "user", "content": MY_QUESTION}]})
    show(mine_result["messages"])

**관찰.** 답의 말투가 내 지침을 따랐는지, 메시지 줄이 앞 셀의 Agent와 어떻게 다른지 읽으세요. 줄 수가 같더라도 마지막 AI 줄의 문장은 달라졌을 것입니다.

## 확인 질문

1. 같은 메뉴 질문이 "도구 없이", "도구 하나", "Agent"에서 각각 무엇을 돌려줬나요? 메뉴 글자는 어느 단계에서 처음 나타났나요?
2. 도구 하나를 묶은 모델 호출에서 `get_menu`를 실행한 쪽은 누구인가요?
3. 시스템 프롬프트를 바꿨을 때 바뀐 것과 바뀌지 않은 것을 메시지 줄로 설명해 보세요.
4. 없는 도구 요청이 들어왔을 때 프로그램은 왜 죽지 않았나요? 실제 모델에게 시켰을 때 예외 없이 끝난 것은 안전한 결과인가요?
5. `recursion_limit`을 줄이면 무엇을 얻고 무엇을 잃나요?
6. 순서가 정해진 일(예: 매일 같은 보고서 만들기)에 Agent를 쓰면 무엇이 나빠지나요?

## 정리

- 모델 호출은 입력 하나에 출력 하나이고, 도구를 묶어도 모델은 **요청**만 한다. 실행은 애플리케이션이 한다.
- Agent는 이 왕복을 harness가 맡는 것이다. 다음 행동은 모델이 고르므로 호출 수와 순서가 실행마다 달라진다.
- 다섯 구성요소 중 모델, 도구, 지침, 상태는 만들 때 넘기고 종료 조건은 실행 설정(`recursion_limit`)으로 정한다.
- 상태는 메시지 목록이다. 한 줄씩 읽을 수 있으면 Agent가 한 일을 설명할 수 있다.
- 없는 도구는 오류 `ToolMessage`로 돌아오고, 끝나지 않는 도구는 한도가 끊는다. 가장 조용한 실패는 도구를 쓰지 않고 그럴듯하게 답하는 것이다.
- Agent를 쓰지 말아야 할 일: 순서가 정해진 일, 되돌릴 수 없는 일, 지연이 엄격한 일.

| 업무 | 고정 workflow / Agent / 둘 다 아님 |
|---|---|
| 매일 같은 형식의 일일 보고서를 만든다 | 고정 workflow |
| 고객 문의를 읽고 필요한 문서를 찾아 답한다 | Agent |
| 한 줄짜리 계산을 한다 | 둘 다 아님 |
| 승인 없이 결제를 취소한다 | 둘 다 아님(사람 승인이 먼저) |

위 분류는 정답이 아니라 토론의 출발점입니다. 내 업무 하나를 같은 기준으로 한 줄 더해 보세요.

**호출 규모.** 이 노트북의 유료 셀은 `announce`가 실행 직전에 추정 횟수를 알려 줍니다. 모든 셀을 한 번씩 돌리면 모델 호출은 수십 회 안팎입니다. 같은 입력도 호출 수가 달라질 수 있습니다.

다음 02에서는 도구 하나의 이름, 설명, 인자 정의가 모델의 선택을 어떻게 바꾸는지, 인자가 틀렸을 때 어떻게 고치게 하는지 다룹니다.